In [ ]:
import pandas as pd

df = pd.read_csv("data/processed/modis/mcd19a2_feature_engineered.csv")

In [ ]:

# Inspect dataset information
df.head(), df.shape, df.columns.tolist(), df.isna().sum().to_dict()

# Correlation Matrix

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Read data from CSV
df = pd.read_csv("data/processed/modis/mcd19a2_feature_engineered.csv")

# Convert the 'date' column to datetime and remove unnecessary columns
df['date'] = pd.to_datetime(df['date'], errors='coerce')  # Convert 'date' to datetime
df = df.select_dtypes(include=[np.number])  # Select numeric columns only

# Compute the correlation matrix
correlation_matrix = df.corr()

# Plot a heatmap to visualize the correlation matrix
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", fmt=".2f", cbar=True)
plt.title('Correlation Matrix Between AOD Features and Atmospheric Indicators')
plt.show()

In [ ]:
# Handle missing values
df_clean = df.dropna(subset=['mcd19a2_aod_055'])  # Remove rows without AOD

# Create X and y
X = df_clean[['aod_ffill', 'aod_rolling3', 'mcd19a2_column_wv']]  # Features
y = df_clean['mcd19a2_aod_055']  # Use AOD 055 as the target

print(f"Shape of X: {X.shape}")
print(f"Shape of y: {y.shape}")

1. Keep missing values unchanged

In [ ]:
# df["aod_available"] = df["mcd19a2_aod_055"].notna().astype(int)

2. Smart fill

In [ ]:
# df = df.sort_values(["station", "date"])

# df["aod_ffill"] = df.groupby("station")["mcd19a2_aod_055"].ffill()
# df["aod_bfill"] = df.groupby("station")["mcd19a2_aod_055"].bfill()

3. Rolling feature (strong signal)

In [ ]:
# df["aod_rolling3"] = df.groupby("station")["mcd19a2_aod_055"].transform(
#     lambda x: x.rolling(3, min_periods=1).mean()
# )

# Step 2: Normalize data for CNN

In [ ]:
from sklearn.preprocessing import StandardScaler

# Normalize data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Recheck
print("X_scaled shape:", X_scaled.shape)

In [ ]:
# Reshape data for CNN (batch_size, height, width, channels)
X_scaled_reshaped = X_scaled.reshape(X_scaled.shape[0], X_scaled.shape[1], 1)

print("X_scaled_reshaped shape:", X_scaled_reshaped.shape)

# Step 3: Split data for CNN

In [ ]:
from sklearn.model_selection import train_test_split

# Split data into train/test (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X_scaled_reshaped, y, test_size=0.2, random_state=42)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")

In [ ]:
# Reshape for CNN: (batch_size, height, width, channels)
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")

# Step 4: Build the CNN model

In [ ]:
# Recheck data after normalization
print(X_scaled.shape)

# Ensure the reshape is correct
X_scaled_reshaped = X_scaled.reshape(X_scaled.shape[0], X_scaled.shape[1], 1)
print(X_scaled_reshaped.shape)  # Recheck shape after reshaping

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.optimizers import Adam

# Build the CNN model with the correct input_shape
model = Sequential([
    Conv2D(32, (2, 2), activation='relu', input_shape=(X_scaled_reshaped.shape[1], 1, 1), padding='same'),
    MaxPooling2D((1, 1), padding='same'),  # Use a smaller kernel (1x1)
    Conv2D(64, (2, 2), activation='relu', padding='same'),
    MaxPooling2D((1, 1), padding='same'),  # Use a smaller kernel (1x1)
    Flatten(),
    Dense(64, activation='relu'),
    Dense(1)  # Predict AOD
])

# Compile model
model.compile(optimizer=Adam(), loss='mean_squared_error', metrics=['mae'])

# Inspect the model
model.summary()

# Step 5: Train the CNN model

In [ ]:
history = model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test))

# Inspect training results
print(f"Final Loss: {history.history['loss'][-1]}")
print(f"Final MAE: {history.history['mae'][-1]}")

# Step 6: Evaluate the model

In [ ]:
import matplotlib.pyplot as plt

# Plot loss
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.legend()
plt.title('Model Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.show()

# Plot MAE
plt.plot(history.history['mae'], label='Training MAE')
plt.plot(history.history['val_mae'], label='Validation MAE')
plt.legend()
plt.title('Model MAE')
plt.xlabel('Epochs')
plt.ylabel('MAE')
plt.show()